<a href="https://colab.research.google.com/github/Najma-1437/Knowledge-Engineering/blob/main/Autonomous_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Practical Exercise 4: Developing a Knowledge-Based Agent

### Knowledge-Based Autonomous Trailer

## Identify the Task

The agent should use knowledge about the road network to choose a route between Nairobi and Kisumu while respecting road availability.

- The agent's environment: a simplified directed Kenyan highway network.
- The goal: identify a minimum-distance permitted route.
- The inputs: current location, destination, road connections and road conditions.
- The outputs: selected route, total distance and an explanation of the decision.

## Assemble Relevant Knowledge


In [27]:
#Highway Network

highway_graph = {
    "Nairobi": {
        "Naivasha": 90,
        "Narok": 142,
        "Thika": 40
    },
    "Naivasha": {
        "Nakuru": 71
    },
    "Narok": {
        "Bomet": 81
    },
    "Thika": {
        "Nyeri": 110
    },
    "Nyeri": {
        "Nyahururu": 103
    },
    "Nyahururu": {
        "Nakuru": 75
    },
    "Nakuru": {
        "Kericho": 108
    },
    "Bomet": {
        "Kericho": 79
    },
    "Kericho": {
        "Kisumu": 84
    },
    "Kisumu": {}
}

# Display the highway network
print("=== HIGHWAY NETWORK KNOWLEDGE BASE ===")

for town, connections in highway_graph.items():
    if connections:
        for destination, distance in connections.items():
            print(f"{town} -> {destination}: {distance} km")
    else:
        print(f"{town}: No outgoing connections")

# Verify the graph
number_of_towns = len(highway_graph)

number_of_roads = sum(
    len(connections)
    for connections in highway_graph.values()
)

print("\n=== GRAPH SUMMARY ===")
print("Number of towns:", number_of_towns)
print("Number of directed roads:", number_of_roads)


=== HIGHWAY NETWORK KNOWLEDGE BASE ===
Nairobi -> Naivasha: 90 km
Nairobi -> Narok: 142 km
Nairobi -> Thika: 40 km
Naivasha -> Nakuru: 71 km
Narok -> Bomet: 81 km
Thika -> Nyeri: 110 km
Nyeri -> Nyahururu: 103 km
Nyahururu -> Nakuru: 75 km
Nakuru -> Kericho: 108 km
Bomet -> Kericho: 79 km
Kericho -> Kisumu: 84 km
Kisumu: No outgoing connections

=== GRAPH SUMMARY ===
Number of towns: 10
Number of directed roads: 11


### Sources of Knowledge
The network consists of 10 towns and 11 directed road connections, with distances measured in kilometres. These distances serve as the edge costs used during route planning.

The graph provides the agent with information about:

- Towns
Road connections
- Distances
- Initial and goal locations

### Assumptions

1. Directed road network: a simplification of the highway network and does not necessarily indicate that the actual roads are one-way.

2. Fixed distances: The distance between two connected towns is assumed to remain constant during route planning.

3. Road availability: All represented roads are initially assumed to be open. Road closures will be introduced as simulated facts to demonstrate knowledge updates and reasoning.

4. Non-negative costs: Every road has a non-negative distance, allowing Uniform Cost Search to determine a minimum-cost route.

5. Simplified environment: The initial knowledge base does not account for live traffic congestion, weather conditions, road accidents or vehicle-specific restrictions.

6. Destination: The autonomous trailer's initial task is to navigate from Nairobi to Kisumu using the available highway network.

The initial implementation uses the highway graph and distance measurements from Practical Exercise 3, Part 1.

No external APIs or web scraping tools are used to construct the initial knowledge base.

In [28]:
print("Number of towns:", len(highway_graph))
print(
    "Number of directed roads:",
    sum(len(neighbors) for neighbors in highway_graph.values())
)

Number of towns: 10
Number of directed roads: 11


##Define the Vocabulary

### Consants
They identify specific objects in the domain. For example, Nairobi, Naivasha, Kericho

### Predicates

| Predicate | Interpretation |
| :--- | :--- |
| `Connected(u, v)` | A directed road connects town *u* to town *v* |
|`Distance(u, v, d)` | The road from *u* to *v* has distance *d* |
| `RoadOpen(u, v)` | The connection from *u* to *v* is open |
|`Usable(u, v)` | The trailer is permitted to travel from *u* to *v* |
| `At(trailer, u)` | The trailer is currently at town *u* |
|`Goal(u)` | Town *u* is the intended destination |


In [29]:
# Initial and goal states
initial_state = "Nairobi"
goal_state = "Kisumu"

print("Initial state:", initial_state)
print("Goal state:", goal_state)

Initial state: Nairobi
Goal state: Kisumu


## Encode General Knowledge About the Domain


### General Rules

| Rule | Meaning |
| :--- | :--- |
| `R1` | A road can be considered only if it exists in the directed graph |
| `R2` | A road can be used only if its status is open |
| `R3` | Every road's traversal cost is its stored distance |
| `R4` | A route's cost is the sum of its road distances |
| `R5` | The agent should select a minimum-cost route among permitted routes |

The central logical rule is:

$$
\text{Connected}(u,v) \land \text{RoadOpen}(u,v) \Rightarrow \text{Usable}(u,v)
$$


in other words, If a directed road connects two towns and that road is open, the trailer may use that connection.


In [30]:
#Road Availability Facts
# All roads are assumed open for the baseline simulation

road_status = {
    (source, destination): "open"
    for source, neighbors in highway_graph.items()
    for destination in neighbors
}
print(road_status[("Nairobi", "Naivasha")])
print("Road status entries:", len(road_status))


open
Road status entries: 11


In [31]:
#Knowledge Base Validation
# Check that the initial and goal towns exist
assert initial_state in highway_graph
assert goal_state in highway_graph

# Check road facts
for source, neighbors in highway_graph.items():
    for destination, distance in neighbors.items():
        assert destination in highway_graph
        assert distance >= 0

# Check that every road has a status
for source, neighbors in highway_graph.items():
    for destination in neighbors:
        assert (source, destination) in road_status

# Check valid road statuses
for status in road_status.values():
    assert status in {"open", "closed"}

print("Knowledge base validation successful.")

Knowledge base validation successful.


## Knowledge Base Interface and Operations

This section implements the ASK and TELL operations that allow the autonomous trailer agent to retrieve  and update information stored in the knowledge base.

In [32]:

class KnowledgeBase:

    def __init__(self, highway_graph, road_status):
        self.graph = highway_graph
        self.road_status = road_status

    # ASK: Retrieve information from the knowledge base
    def ask(self, predicate, source, destination):

        if predicate == "Connected":
            return destination in self.graph.get(source, {})

        elif predicate == "Distance":
            return self.graph.get(source, {}).get(destination)

        elif predicate == "RoadStatus":
            return self.road_status.get((source, destination))

        elif predicate == "RoadOpen":
            status = self.road_status.get((source, destination))
            return status == "open" if status is not None else False

        else:
            raise ValueError(f"Unknown predicate: {predicate}")

    # TELL: Add or update information in the knowledge base
    def tell(self, predicate, source, destination, value):

        if predicate != "RoadStatus":
            raise ValueError("Only RoadStatus updates are supported.")

        if destination not in self.graph.get(source, {}):
            raise ValueError("The specified road does not exist.")

        if value not in {"open", "closed"}:
            raise ValueError("Road status must be 'open' or 'closed'.")

        self.road_status[(source, destination)] = value

        return f"Knowledge updated: {source} -> {destination} is {value}."


In [33]:

# Create an independent copy for testing
from copy import deepcopy

test_graph = deepcopy(highway_graph)
test_status = deepcopy(road_status)

# Initialise the knowledge base
kb = KnowledgeBase(test_graph, test_status)

# Test ASK operations
print("=== ASK OPERATION TESTS ===")

print("Road connected:",
      kb.ask("Connected", "Nairobi", "Naivasha"))

print("Road distance:",
      kb.ask("Distance", "Nairobi", "Naivasha"), "km")

print("Road status:",
      kb.ask("RoadStatus", "Nairobi", "Naivasha"))

print("Road usable status:",
      kb.ask("RoadOpen", "Nairobi", "Naivasha"))

# Test TELL operation
print("\n=== TELL OPERATION TEST ===")

result = kb.tell(
    "RoadStatus",
    "Nairobi",
    "Naivasha",
    "closed"
)

print(result)

# Verify that the knowledge was updated
print("\nUpdated road status:",
      kb.ask("RoadStatus", "Nairobi", "Naivasha"))

print("Can trailer use road:",
      kb.ask("RoadOpen", "Nairobi", "Naivasha"))

# Verify that the original knowledge remains unchanged
assert road_status[("Nairobi", "Naivasha")] == "open"

# Verify that the test knowledge was updated
assert kb.ask("RoadStatus", "Nairobi", "Naivasha") == "closed"
assert kb.ask("RoadOpen", "Nairobi", "Naivasha") is False

print("\nAll ASK and TELL tests passed.")


=== ASK OPERATION TESTS ===
Road connected: True
Road distance: 90 km
Road status: open
Road usable status: True

=== TELL OPERATION TEST ===
Knowledge updated: Nairobi -> Naivasha is closed.

Updated road status: closed
Can trailer use road: False

All ASK and TELL tests passed.


In [34]:
class InferenceEngine:

    def __init__(self, knowledge_base):
        self.kb = knowledge_base

    def is_road_usable(self, source, destination):
        pass

## Inference Engine

This section implements the inference rules used to determine whether a road can be used by the autonomous trailer.

In [35]:
class InferenceEngine:

    def __init__(self, knowledge_base):
        self.kb = knowledge_base

    # Apply the rule:
    # Connected(u, v) AND RoadOpen(u, v) => Usable(u, v)
    def is_road_usable(self, source, destination):

        connected = self.kb.ask(
            "Connected", source, destination
        )

        road_open = self.kb.ask(
            "RoadOpen", source, destination
        )

        return connected and road_open

    # Construct a graph containing only usable roads
    def build_usable_graph(self):

        usable_graph = {
            town: {} for town in self.kb.graph
        }

        for source, destinations in self.kb.graph.items():

            for destination, distance in destinations.items():

                if self.is_road_usable(source, destination):
                    usable_graph[source][destination] = distance

        return usable_graph

In [36]:
# Initialise the inference engine
inference_engine = InferenceEngine(kb)

print("=== INFERENCE ENGINE TESTS ===")

# Test a road that was closed using TELL
print(
    "Nairobi -> Naivasha usable:",
    inference_engine.is_road_usable(
        "Nairobi", "Naivasha"
    )
)

# Test a road that remains open
print(
    "Nairobi -> Narok usable:",
    inference_engine.is_road_usable(
        "Nairobi", "Narok"
    )
)

# Build the graph containing only usable roads
usable_graph = inference_engine.build_usable_graph()

# Count usable roads
usable_road_count = sum(
    len(destinations)
    for destinations in usable_graph.values()
)

print("\nUsable road connections:", usable_road_count)
print("Original road connections:", 11)

# Verify the inference results
assert inference_engine.is_road_usable(
    "Nairobi", "Naivasha"
) is False

assert inference_engine.is_road_usable(
    "Nairobi", "Narok"
) is True

assert usable_road_count == 10

print("\nAll inference engine tests passed.")

=== INFERENCE ENGINE TESTS ===
Nairobi -> Naivasha usable: False
Nairobi -> Narok usable: True

Usable road connections: 10
Original road connections: 11

All inference engine tests passed.


In [37]:
import heapq
import itertools

### UCS trailer implementation from Ex.3

In [38]:
def uniform_cost_search(graph, start, goal, verbose=True):

    counter = itertools.count()
    frontier = [(0, next(counter), start)]

    best_cost = {start: 0}
    parent = {start: None}

    explored = set()
    expanded_order = []

    step = 0

    while frontier:

        g, _, node = heapq.heappop(frontier)

        if node in explored:
            continue

        step += 1
        explored.add(node)
        expanded_order.append(node)

        if verbose:
            print(f"Step {step}: expand {node} (g = {g} km)")

        if node == goal:

            path = []
            current = node

            while current is not None:
                path.append(current)
                current = parent[current]

            path.reverse()

            if verbose:
                print(f"\nGoal '{goal}' reached.")

            return path, g, expanded_order

        for neighbor, step_cost in graph[node].items():

            new_cost = g + step_cost

            if (neighbor not in explored and
                    new_cost < best_cost.get(neighbor, float("inf"))):

                best_cost[neighbor] = new_cost
                parent[neighbor] = node

                heapq.heappush(
                    frontier,
                    (new_cost, next(counter), neighbor)
                )

        if verbose:

            live = sorted(
                (c, n) for c, _, n in frontier
                if n not in explored and c == best_cost[n]
            )

            print("   Frontier:", [(n, c) for c, n in live])

    return None, float("inf"), expanded_order

In [39]:
print("=== UCS INTEGRATION TEST ===")

path, total_cost, expanded = uniform_cost_search(
    usable_graph,
    initial_state,
    goal_state
)

if path is None:
    print("No permitted route is available.")

else:
    print("\nSelected route:", " -> ".join(path))
    print("Total distance:", total_cost, "km")
    print("Expanded nodes:", expanded)

=== UCS INTEGRATION TEST ===
Step 1: expand Nairobi (g = 0 km)
   Frontier: [('Thika', 40), ('Narok', 142)]
Step 2: expand Thika (g = 40 km)
   Frontier: [('Narok', 142), ('Nyeri', 150)]
Step 3: expand Narok (g = 142 km)
   Frontier: [('Nyeri', 150), ('Bomet', 223)]
Step 4: expand Nyeri (g = 150 km)
   Frontier: [('Bomet', 223), ('Nyahururu', 253)]
Step 5: expand Bomet (g = 223 km)
   Frontier: [('Nyahururu', 253), ('Kericho', 302)]
Step 6: expand Nyahururu (g = 253 km)
   Frontier: [('Kericho', 302), ('Nakuru', 328)]
Step 7: expand Kericho (g = 302 km)
   Frontier: [('Nakuru', 328), ('Kisumu', 386)]
Step 8: expand Nakuru (g = 328 km)
   Frontier: [('Kisumu', 386)]
Step 9: expand Kisumu (g = 386 km)

Goal 'Kisumu' reached.

Selected route: Nairobi -> Narok -> Bomet -> Kericho -> Kisumu
Total distance: 386 km
Expanded nodes: ['Nairobi', 'Thika', 'Narok', 'Nyeri', 'Bomet', 'Nyahururu', 'Kericho', 'Nakuru', 'Kisumu']


### Experiment A: Agent recalculates after a selected road closes

In [40]:
from copy import deepcopy

# Create an independent knowledge base for Experiment A
experiment_a_graph = deepcopy(highway_graph)
experiment_a_status = deepcopy(kb.road_status)

kb_a = KnowledgeBase(
    experiment_a_graph,
    experiment_a_status
)

# Close Nairobi -> Narok
print("=== EXPERIMENT A: ROAD CLOSURE ===")

print(kb_a.tell(
    "RoadStatus",
    "Nairobi",
    "Narok",
    "closed"
))

# Rebuild the inference engine and usable graph
inference_a = InferenceEngine(kb_a)
usable_graph_a = inference_a.build_usable_graph()

# Recalculate the route using UCS
path_a, cost_a, expanded_a = uniform_cost_search(
    usable_graph_a,
    initial_state,
    goal_state,
    verbose=False
)

print("\n=== RECALCULATED ROUTE ===")

if path_a is not None:
    print("Selected route:", " -> ".join(path_a))
    print("Total distance:", cost_a, "km")
else:
    print("No permitted route is available.")

# Verify the result
assert path_a == [
    "Nairobi",
    "Thika",
    "Nyeri",
    "Nyahururu",
    "Nakuru",
    "Kericho",
    "Kisumu"
]

assert cost_a == 520

print("\nExperiment A passed.")

=== EXPERIMENT A: ROAD CLOSURE ===
Knowledge updated: Nairobi -> Narok is closed.

=== RECALCULATED ROUTE ===
Selected route: Nairobi -> Thika -> Nyeri -> Nyahururu -> Nakuru -> Kericho -> Kisumu
Total distance: 520 km

Experiment A passed.


### Experiment B: Close another road and agent recalculates

In [41]:

# =====================================================
# EXPERIMENT B: CLOSE KERICHO -> KISUMU
# =====================================================

print("=== EXPERIMENT B: ROAD CLOSURE ===")

# Step 1: Update the knowledge base
result = kb.tell(
    "RoadStatus",
    "Kericho",
    "Kisumu",
    "closed"
)

print(result)

# Step 2: Rebuild the usable graph using updated knowledge
usable_graph_B = inference_engine.build_usable_graph()

print("\n=== UPDATED ROAD STATUS ===")

print(
    "Kericho -> Kisumu usable:",
    inference_engine.is_road_usable(
        "Kericho", "Kisumu"
    )
)

# Step 3: Recalculate the route using UCS
print("\n=== EXPERIMENT B: UCS ROUTE RECALCULATION ===")

path_B, total_cost_B, expanded_B = uniform_cost_search(
    usable_graph_B,
    initial_state,
    goal_state,
    verbose=True
)

# Step 4: Display the result
print("\n=== EXPERIMENT B RESULT ===")

if path_B is None:
    print("No permitted route is available.")
    print("Destination:", goal_state)
    print("Reason: Kericho -> Kisumu is closed.")
else:
    print("Selected route:", " -> ".join(path_B))
    print("Total distance:", total_cost_B, "km")

# Step 5: Verify the expected result
assert path_B is None, "Expected no permitted route to Kisumu."

print("\nExperiment B passed.")


=== EXPERIMENT B: ROAD CLOSURE ===
Knowledge updated: Kericho -> Kisumu is closed.

=== UPDATED ROAD STATUS ===
Kericho -> Kisumu usable: False

=== EXPERIMENT B: UCS ROUTE RECALCULATION ===
Step 1: expand Nairobi (g = 0 km)
   Frontier: [('Thika', 40), ('Narok', 142)]
Step 2: expand Thika (g = 40 km)
   Frontier: [('Narok', 142), ('Nyeri', 150)]
Step 3: expand Narok (g = 142 km)
   Frontier: [('Nyeri', 150), ('Bomet', 223)]
Step 4: expand Nyeri (g = 150 km)
   Frontier: [('Bomet', 223), ('Nyahururu', 253)]
Step 5: expand Bomet (g = 223 km)
   Frontier: [('Nyahururu', 253), ('Kericho', 302)]
Step 6: expand Nyahururu (g = 253 km)
   Frontier: [('Kericho', 302), ('Nakuru', 328)]
Step 7: expand Kericho (g = 302 km)
   Frontier: [('Nakuru', 328)]
Step 8: expand Nakuru (g = 328 km)
   Frontier: []

=== EXPERIMENT B RESULT ===
No permitted route is available.
Destination: Kisumu
Reason: Kericho -> Kisumu is closed.

Experiment B passed.


In [42]:

def explain_route(path, total_distance, knowledge_base):

    print("=== AGENT DECISION EXPLANATION ===")

    # Case 1: No route was found
    if path is None:

        print("Decision: No permitted route is available.")

        print(
            "Explanation: The destination cannot be reached "
            "using the currently open roads."
        )

        closed_roads = [
            (source, destination)
            for (source, destination), status
            in knowledge_base.road_status.items()
            if status == "closed"
        ]

        if closed_roads:
            print("\nClosed roads in the knowledge base:")

            for source, destination in closed_roads:
                print(f"- {source} -> {destination}")

        return

    # Case 2: Verify that every road in the route is usable
    route_is_valid = True
    calculated_distance = 0

    print("\nChecking selected route:")

    for source, destination in zip(path, path[1:]):

        status = knowledge_base.ask(
            "RoadStatus", source, destination
        )

        distance = knowledge_base.ask(
            "Distance", source, destination
        )

        if status != "open" or distance is None:

            print(
                f"- {source} -> {destination}: "
                f"NOT USABLE ({status})"
            )

            route_is_valid = False

        else:

            print(
                f"- {source} -> {destination}: "
                f"{distance} km (open)"
            )

            calculated_distance += distance

    # Case 3: Reject a stale or invalid route
    if not route_is_valid:

        print("\nDecision: Previously selected route is no longer valid.")
        print(
            "Explanation: At least one road on the route "
            "is currently closed or unavailable."
        )

        return

    if calculated_distance != total_distance:

        print("\nWarning: The supplied route distance does not match "
              "the current knowledge base.")

        return

    # Case 4: Explain a valid route
    print("\nDecision: A permitted route was found.")

    print("Selected route:", " -> ".join(path))

    print("Total distance:", total_distance, "km")

    print(
        "\nExplanation: All roads on the selected route are open. "
        "UCS selected this route as the minimum-cost route "
        "in the current usable graph."
    )


### Explanation of Experiment B

In [43]:
explain_route(
    path_B,
    total_cost_B,
    kb
)

=== AGENT DECISION EXPLANATION ===
Decision: No permitted route is available.
Explanation: The destination cannot be reached using the currently open roads.

Closed roads in the knowledge base:
- Nairobi -> Naivasha
- Kericho -> Kisumu


### Explanation of Experiment A


In [44]:
explain_route(
    path,
    total_cost,
    kb
)

=== AGENT DECISION EXPLANATION ===

Checking selected route:
- Nairobi -> Narok: 142 km (open)
- Narok -> Bomet: 81 km (open)
- Bomet -> Kericho: 79 km (open)
- Kericho -> Kisumu: NOT USABLE (closed)

Decision: Previously selected route is no longer valid.
Explanation: At least one road on the route is currently closed or unavailable.


## Implementation Summary

The implementation of the inference engine and autonomous trailer agent was completed successfully.

The Knowledge Base Interface was implemented using ASK and TELL operations, enabling the agent to retrieve road information and update road conditions dynamically.

An inference engine was developed to determine road usability using the rule that a road is usable only if it exists in the highway network and is currently open.

The inference engine was integrated with Uniform Cost Search (UCS) to identify the minimum-distance route using only available roads.

Dynamic road closure experiments demonstrated the agent's ability to update its knowledge, reconstruct the usable road network and recalculate routes. The experiments produced the following results:

- Initial route: 353 km.
- After closing Nairobi–Naivasha: 386 km.
- After subsequently closing Nairobi–Narok: 520 km.
- After subsequently closing Kericho–Kisumu: No permitted route available.

Finally, a route explanation function was implemented to communicate the agent's decisions, selected routes, distances and reasons why a destination may be unreachable.

These results demonstrate the successful implementation of knowledge querying, knowledge updating, rule-based inference, cost-based route selection and dynamic decision-making.

In [45]:
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\david\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [46]:
import ipywidgets as widgets
from IPython.display import display, clear_output

print("Interactive widgets are ready!")

Interactive widgets are ready!


In [47]:
print("Highway graph available:", "highway_graph" in globals())

print("\nKnowledge Base objects:")
for name, obj in globals().copy().items():
    if obj.__class__.__name__ == "KnowledgeBase":
        print(name)

print("\nInference Engine objects:")
for name, obj in globals().copy().items():
    if obj.__class__.__name__ == "InferenceEngine":
        print(name)

print("\nAvailable towns:")
print(list(highway_graph.keys()))

Highway graph available: True

Knowledge Base objects:
kb
kb_a

Inference Engine objects:
inference_engine
inference_a
engine

Available towns:
['Nairobi', 'Naivasha', 'Narok', 'Thika', 'Nyeri', 'Nyahururu', 'Nakuru', 'Bomet', 'Kericho', 'Kisumu']


In [51]:

import ipywidgets as widgets
from IPython.display import display, clear_output
import math

# Find the existing Knowledge Base and Inference Engine
kb_objects = [
    obj for obj in globals().values()
    if isinstance(obj, KnowledgeBase)
]

engine_objects = [
    obj for obj in globals().values()
    if isinstance(obj, InferenceEngine)
]

if not kb_objects or not engine_objects:
    raise RuntimeError(
        "Knowledge Base or Inference Engine not found. "
        "Run the existing notebook cells first."
    )

# Use the Knowledge Base connected to the Inference Engine
engine = engine_objects[0]
kb = engine.kb

towns = list(kb.graph.keys())

# -----------------------------
# ROUTE SELECTION CONTROLS
# -----------------------------

start_dropdown = widgets.Dropdown(
    options=towns,
    value=towns[0],
    description="Start:"
)

goal_dropdown = widgets.Dropdown(
    options=towns,
    value=towns[-1],
    description="Destination:"
)

find_button = widgets.Button(
    description="Find Best Route",
    button_style="success",
    icon="search"
)

route_output = widgets.Output()

# -----------------------------
# ROAD UPDATE CONTROLS
# -----------------------------

road_source = widgets.Dropdown(
    options=[town for town in towns if kb.graph[town]],
    description="From:"
)

def get_destinations(source):
    return list(kb.graph[source].keys())

road_destination = widgets.Dropdown(
    options=get_destinations(road_source.value),
    description="To:"
)

road_status_dropdown = widgets.Dropdown(
    options=["open", "closed"],
    value="closed",
    description="Status:"
)

update_button = widgets.Button(
    description="Update Road",
    button_style="warning",
    icon="refresh"
)

update_output = widgets.Output()

# Change destination options when road source changes
def update_destinations(change):
    destinations = get_destinations(change["new"])
    road_destination.options = destinations
    if destinations:
        road_destination.value = destinations[0]

road_source.observe(update_destinations, names="value")

# -----------------------------
# UPDATE KNOWLEDGE BASE
# -----------------------------

def update_road(button):
    with update_output:
        clear_output()

        source = road_source.value
        destination = road_destination.value
        status = road_status_dropdown.value

        try:
            message = kb.tell(
                "RoadStatus",
                source,
                destination,
                status
            )
            print(message)

        except ValueError as error:
            print("Update failed:", error)

update_button.on_click(update_road)

# -----------------------------
# FIND BEST ROUTE USING UCS
# -----------------------------

def find_route(button):
    with route_output:
        clear_output()

        start = start_dropdown.value
        goal = goal_dropdown.value

        if start == goal:
            print("Start and destination are the same town.")
            print("Total distance: 0 km")
            return

        # Build graph containing only usable roads
        usable_graph = engine.build_usable_graph()

        # Run Uniform Cost Search
        path, distance, expanded = uniform_cost_search(
            usable_graph,
            start,
            goal,
            verbose=False
        )

        print("AUTONOMOUS TRAILER ROUTE REPORT")
        print("--------------------------------")

        if path is None:
            print(f"No available route from {start} to {goal}.")
            print("\nReason:")
            print("The destination cannot be reached using open roads.")

        else:
            print("Recommended route:")
            print(" -> ".join(path))

            print(f"\nTotal distance: {distance} km")

            print("\nRoute explanation:")
            for i in range(len(path) - 1):
                source = path[i]
                destination = path[i + 1]
                road_distance = kb.ask(
                    "Distance", source, destination
                )
                status = kb.ask(
                    "RoadStatus", source, destination
                )

                print(
                    f"{source} -> {destination}: "
                    f"{road_distance} km, status: {status}"
                )

        print("\nExpanded towns:")
        print(", ".join(expanded))

find_button.on_click(find_route)

# -----------------------------
# DISPLAY INTERACTIVE INTERFACE
# -----------------------------

print("AUTONOMOUS TRAILER AGENT")
print("========================")

display(
    widgets.VBox([
        widgets.HTML("<h3>1. Select Your Journey</h3>"),
        start_dropdown,
        goal_dropdown,
        find_button,
        route_output,

        widgets.HTML("<hr><h3>2. Update Road Conditions</h3>"),
        road_source,
        road_destination,
        road_status_dropdown,
        update_button,
        update_output
    ])
)


AUTONOMOUS TRAILER AGENT


In [49]:
# Reset all roads to open using the Knowledge Base
for source, destinations in kb.graph.items():
    for destination in destinations:
        kb.tell("RoadStatus", source, destination, "open")

print("All roads have been reopened successfully!")

All roads have been reopened successfully!


## User Interface, Testing and Documentation

### 1. User Interface Development

An interactive user interface was developed using Python's `ipywidgets` library in Jupyter Notebook. The interface allows users to select a starting town and destination, update road conditions, and request the autonomous trailer agent to calculate the best available route.

### 2. System Integration

The interface was integrated with the Knowledge Base, Inference Engine and Uniform Cost Search algorithm. This enables users to update road information dynamically and obtain route recommendations based on the current road conditions.

### 3. Testing and Results

Three test scenarios were conducted:

| Test | Scenario                | Result                      |
| ---- | ----------------------- | --------------------------- |
| 1    | All roads open          | Optimal route of 353 km     |
| 2    | Nairobi–Naivasha closed | Alternative route of 386 km |
| 3    | Kericho–Kisumu closed   | No available route          |

**Figure 1: Interactive interface showing the alternative route after Nairobi–Naivasha was closed.**

The agent selected:

Nairobi → Narok → Bomet → Kericho → Kisumu

Total distance: **386 km**.

The screenshot demonstrates that the system successfully updated the road condition and recalculated the route using the available roads.

### 4. Conclusion

The interactive system successfully demonstrates autonomous route planning using knowledge representation, inference and Uniform Cost Search. It responds to changes in road conditions by recalculating routes and identifying situations where the destination is unreachable.

### 5. Limitations

The system uses a predefined highway network and manually updated road conditions. It does not currently receive live traffic information or automatically detect road closures.
